# Anti Money Laudenring Study
---

The primary objective of this analysis is to develop a machine learning model capable of identifying potentially suspicious transactions or customers for AML purposes.

We aim to perform data preprocessing and feature engineering to create meaningful inputs for modeling and then train and evaluate a machine learning model to detect suspicious behavior. Finally, we will provide interpretable insights that can support AML analysts and compliance decision-making.

Follow bellow some references for this study:

#### Clustering 

[1] Chandola, Varun, Arindam Banerjee, and Vipin Kumar. "Anomaly detection: A survey." ACM computing surveys (CSUR) 41.3 (2009): 1-58. Access on: https://dl.acm.org/doi/abs/10.1145/1541880.1541882

[2] Knox, Edwin M., and Raymond T. Ng. "Algorithms for mining distancebased outliers in large datasets." Proceedings of the international conference on very large data bases. Citeseer, 1998. Access on: https://www.researchgate.net/publication/2346156_Algorithms_for_Mining_Distance-Based_Outliers_in_Large_Datasets

[3] Jensen, Rasmus Ingemann Tuffveson, and Alexandros Iosifidis. "Fighting money laundering with statistics and machine learning." Ieee Access 11 (2023): 8889-8903. Access on: https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=10025710

#### Dimension reduction

[4] Bakry, Ahmed N., et al. "Combating financial crimes with unsupervised learning techniques: clustering and dimensionality reduction for anti-money laundering." arXiv preprint arXiv:2403.00777 (2024). Access on: https://arxiv.org/pdf/2403.00777


### Imports
---

In [1]:
# system
import os
import sys
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(ROOT_PATH)

# utils
from datetime import date, timedelta
import yfinance as yf

#visualization
import plotly.express as px

# data manipulation
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', None)

# personalized modules
from src.utils.dataset import get_raw_transactions

2026-04-02 22:28:36.628 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-02 22:28:36.631 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-02 22:28:36.632 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
random_state = 42

### Load data
---

To get started, we are going to load data from IBM dataset available on Kaggle. You can get the data on the following link: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml.
After downloading it, make sure that the data is available in the right path "src/data/HI-Small_Trans.csv"

In [3]:
data_path = f"{ROOT_PATH}\\src\\data\\HI-Small_Trans.csv"
transactions_df = get_raw_transactions(data_path)

In [4]:
transactions_df.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,"3,697.34",US Dollar,"3,697.34",US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,"14,675.57",US Dollar,"14,675.57",US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,"2,806.97",US Dollar,"2,806.97",US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,"36,682.97",US Dollar,"36,682.97",US Dollar,Reinvestment,0


### Transformations
---

In this step, the goal is to transform the original transaction-level dataset into a customer-level dataset. Since the objective is to cluster customers and assign a risk score at the customer level, we must first aggregate transactional information into meaningful customer-based features.

To uniquely identify each customer, we define a composite ID based on the combination of account and bank, ensuring consistent identification across the dataset.

Additionally, because transactions may occur in different currencies, it is necessary to standardize all monetary values into a single reference currency. To ensure comparability across the entire customer base, all transaction amounts will be converted to USD. For greater consistency and accuracy, the conversion can be performed using the exchange rate corresponding to the transaction date.

In [5]:
transformed_df = transactions_df.copy()
transformed_df.columns = [col.strip().lower().replace(" ", "_") for col in transformed_df.columns]

# create unique customer identifier
transformed_df["sender_customer"] = transformed_df["account"].astype(str) + "_" + transformed_df["from_bank"].astype(str)
transformed_df["receiver_customer"] = transformed_df["account.1"].astype(str) + "_" + transformed_df["to_bank"].astype(str)

#remove transactions with destination equal to the origin
transformed_df = transformed_df[transformed_df["payment_format"] != "Reinvestment"]

transformed_df["timestamp"] = pd.to_datetime(transformed_df['timestamp'])

In [6]:
currency2tickers = {
    "us dollar": "usdusd",
    "euro": "eurusd",
    "swiss franc": "chfusd",
    "yuan": "cnyusd",
    "shekel": "ilsusd",
    "rupee": "inrusd",
    "uk pound": "gbpusd",
    "ruble": "rubusd",
    "yen": "jpyusd",
    "bitcoin": "btc-usd",
    "canadian dollar": "cadusd",
    "australian dollar": "audusd",
    "mexican peso": "mxnusd",
    "saudi riyal": "sarusd",
    "brazil real": "brlusd"
}
tickers2currency = {v: k for k, v in currency2tickers.items()}
tickers = [
    "EURUSD=X", "CHFUSD=X", "CNYUSD=X", "ILSUSD=X", "INRUSD=X",
    "GBPUSD=X", "RUBUSD=X", "JPYUSD=X", "CADUSD=X", "AUDUSD=X",
    "MXNUSD=X", "SARUSD=X", "BRLUSD=X", "BTC-USD"
]
start_date = date.today() - timedelta(days=7)
currency_conversion = yf.download(tickers=tickers, start=start_date)
currency_conversion = currency_conversion["Close"].reset_index()
currency_conversion.columns = [column.lower().replace("=x", '') for column in currency_conversion.columns]

currency_conversion = currency_conversion.dropna().iloc[0:1, :]
currency_conversion["usdusd"] = 1.0
currency_conversion = currency_conversion.melt(id_vars=["date"], var_name="currency", value_name="exchange_rate")
currency_conversion["currency_name"] = currency_conversion["currency"].apply(lambda x: tickers2currency.get(x, "unknown"))
currency_conversion = currency_conversion.drop(columns=["date"])

C:\Users\ferna\AppData\Local\Temp\ipykernel_8800\1770237953.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  currency_conversion = yf.download(tickers=tickers, start=start_date)
[*********************100%***********************]  14 of 14 completed


In [7]:
transformed_df["payment_currency"] = transformed_df["payment_currency"].str.lower().str.strip()
transformed_df=transformed_df.merge(currency_conversion, left_on="payment_currency", right_on="currency_name", how="left")
transformed_df["usd_amount_paid"] = transformed_df["amount_paid"] * transformed_df["exchange_rate"]

In [8]:
transformed_df['transaction_date'] = transformed_df["timestamp"].dt.date
transformed_df["transaction_date"] = pd.to_datetime(transformed_df["transaction_date"])

In [9]:
transformed_df = (
    transformed_df[["timestamp", "transaction_date", "sender_customer", "receiver_customer", "usd_amount_paid"]]
    .rename(columns={"usd_amount_paid": "amount"})    
)

In [10]:
transformed_df.head()

,timestamp,transaction_date,sender_customer,receiver_customer,amount
0,2022-09-01 00:20:00,2022-09-01,8000F4580_3208,8000F5340_1,0.01
1,2022-09-01 00:26:00,2022-09-01,8000EC280_12,8017BF800_2439,7.66
2,2022-09-01 00:21:00,2022-09-01,8000EDEC0_1,80AEF5310_211050,383.71
3,2022-09-01 00:04:00,2022-09-01,8000F4510_1,8011305D0_11813,9.82
4,2022-09-01 00:08:00,2022-09-01,8000F4FE0_1,812ED62E0_245335,4.01


In [11]:
ROOT_PATH

'c:\\Users\\ferna\\Documents\\anti-money-laundering-analysis'

In [12]:
transformed_df.to_csv(f"{ROOT_PATH}/src/data/output/raw_transactions.csv", index=False)

### Feature Engineering
---
In this section, we build the customer-level features that will be used in the AML model. Since our objective is to assess customer risk, transactional data is aggregated into behavioral indicators that summarize activity patterns.

The features are organized into three main groups:

1. **Transaction Behavior**: overall activity metrics such as volume, counts, and average amounts.

2. **Time Variance**: features capturing changes and patterns over time.

3. **Ratio Metrics**: proportional indicators, such as inflow/outflow relationships and concentration measures.

This structure allows the model to capture both activity intensity and behavioral patterns.

#### Transaction behavior features
---

In [13]:
sent_df = transformed_df.assign(
    customer_id=transformed_df["sender_customer"],
    direction="sent",
    amount=transformed_df["amount"],
    tx_count=1
)[["customer_id", "transaction_date", "direction", "amount", "tx_count"]]

received_df = transformed_df.assign(
    customer_id=transformed_df["receiver_customer"],
    direction="received",
    amount=transformed_df["amount"],
    tx_count=1
)[["customer_id", "transaction_date", "direction", "amount", "tx_count"]]

simplified_transaction_df = pd.concat([sent_df, received_df], ignore_index=True)

In [14]:
simplified_transaction_df.head()

,customer_id,transaction_date,direction,amount,tx_count
0,8000F4580_3208,2022-09-01,sent,0.01,1
1,8000EC280_12,2022-09-01,sent,7.66,1
2,8000EDEC0_1,2022-09-01,sent,383.71,1
3,8000F4510_1,2022-09-01,sent,9.82,1
4,8000F4FE0_1,2022-09-01,sent,4.01,1


In [15]:
transaction_behavior_df = (
    simplified_transaction_df.groupby(by=["customer_id", "direction"], as_index=False)
    .agg(
        transaction_count=("amount", "count"),
        total_amount=("amount", "sum"),
        median_amount=("amount", "median"),
        std_amount=("amount", "std"),
        max_amount=("amount", "max"),
    )
    .reset_index(drop=True)
    .pivot(index="customer_id", columns="direction")
    .fillna(0)
)
transaction_behavior_df.columns = [
    f"{metric}_{direction}"
    for metric, direction in transaction_behavior_df.columns
]
transaction_behavior_df = transaction_behavior_df.reset_index()

In [16]:
transaction_behavior_df.head()

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,max_amount_sent
0,100428660_70,"1,084.00","168,672.00","479,594.79","52,762,291,209.75",238.44,"1,061.85",637.65,"8,093,751.98","5,551.91","1,615,849,280.83"
1,1004286A8_70,653.00,"103,018.00","290,195.02","30,140,098,702.43",261.51,989.80,673.99,"6,795,155.33","4,971.15","954,757,442.23"
2,1004286F0_70,108.00,"18,663.00","36,796.68","3,573,012,238.98",188.05,910.14,418.88,"2,869,372.33","2,117.38","165,255,167.69"
3,100428738_70,98.00,"13,756.00","39,554.71","1,626,366,355.50",200.39,690.23,598.17,"2,625,842.87","2,410.15","187,245,987.35"
4,100428780_70,117.00,"17,264.00","111,632.45","3,948,876,147.96",217.21,767.33,"4,371.38","4,548,156.67","33,667.82","286,863,871.15"


#### Time features
---

In [17]:
simplified_time_df = (
    simplified_transaction_df
    .groupby(by=["customer_id", "transaction_date", "direction"])
    .agg(
        total_amount=("amount", "sum"),
        transaction_count=("tx_count", "sum"),
    )
    .reset_index()
)

reference_date_df = (
    simplified_time_df
    .groupby(by="customer_id", as_index=False)
    .agg(
        reference_date=("transaction_date", "max")
    )
)

simplified_time_df = simplified_time_df.merge(reference_date_df, on="customer_id", how="left", validate="many_to_one")

In [18]:
simplified_time_df

,customer_id,transaction_date,direction,total_amount,transaction_count,reference_date
0,100428660_70,2022-09-01,received,0.19,6,2022-09-10
1,100428660_70,2022-09-01,sent,"13,482,513,260.41",22801,2022-09-10
2,100428660_70,2022-09-02,received,"239,797.30",539,2022-09-10
3,100428660_70,2022-09-02,sent,"15,557,700,792.93",26365,2022-09-10
4,100428660_70,2022-09-03,sent,"883,461,109.84",7440,2022-09-10
...,...,...,...,...,...,...
3013442,814965B51_255063,2022-09-07,sent,"193,765.51",5,2022-09-09
3013443,814965B51_255063,2022-09-08,received,"193,765.51",5,2022-09-09
3013444,814965B51_255063,2022-09-08,sent,"193,765.51",5,2022-09-09
3013445,814965B51_255063,2022-09-09,received,"193,765.51",5,2022-09-09


In [19]:
simplified_time_df["days_from_ref"] = (simplified_time_df["reference_date"] - simplified_time_df["transaction_date"]).dt.days

In [20]:
bins = [-1, 7, 30, 90]
labels = ["7d", "30d", "90d"]
expected_windows = ["7d", "30d", "90d"]
expected_directions = simplified_time_df["direction"].unique()
expected_metrics = ["total_amount", "transaction_count"]
multi_cols = pd.MultiIndex.from_product(
    [expected_metrics, expected_directions, expected_windows]
)

simplified_time_df["window"] = pd.cut(
    simplified_time_df["days_from_ref"],
    bins=bins,
    labels=labels
)

In [21]:
simplified_time_df["window"] = simplified_time_df["window"].astype(str)
time_behavior_df = (
    simplified_time_df
    .groupby(["customer_id", "direction", "window"], as_index=False)
    .agg(
        total_amount=("total_amount", "sum"),
        transaction_count=("transaction_count", "sum")
    )
    .pivot_table(
        index="customer_id",
        columns=["direction", "window"],
        values=["total_amount", "transaction_count"],
        fill_value=0
    )
)
time_behavior_df = time_behavior_df.reindex(columns=multi_cols, fill_value=0)
time_behavior_df.columns = [f"{metric}_{direction}_{window}" for metric, direction, window in time_behavior_df.columns]
time_behavior_df = time_behavior_df.reset_index()

#### Ratio features
---

In [22]:
transaction_behavior_df

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,max_amount_sent
0,100428660_70,"1,084.00","168,672.00","479,594.79","52,762,291,209.75",238.44,"1,061.85",637.65,"8,093,751.98","5,551.91","1,615,849,280.83"
1,1004286A8_70,653.00,"103,018.00","290,195.02","30,140,098,702.43",261.51,989.80,673.99,"6,795,155.33","4,971.15","954,757,442.23"
2,1004286F0_70,108.00,"18,663.00","36,796.68","3,573,012,238.98",188.05,910.14,418.88,"2,869,372.33","2,117.38","165,255,167.69"
3,100428738_70,98.00,"13,756.00","39,554.71","1,626,366,355.50",200.39,690.23,598.17,"2,625,842.87","2,410.15","187,245,987.35"
4,100428780_70,117.00,"17,264.00","111,632.45","3,948,876,147.96",217.21,767.33,"4,371.38","4,548,156.67","33,667.82","286,863,871.15"
...,...,...,...,...,...,...,...,...,...,...,...
425110,814965930_27995,23.00,2.00,"6,965,163.49","4,630.75","354,368.05","2,315.37","263,137.55","2,190.38","627,052.98","3,864.20"
425111,8149659D0_36868,24.00,24.00,"454,724.23","454,724.23","3,776.05","3,776.05","48,454.74","48,454.74","173,995.72","173,995.72"
425112,814965AB0_245748,14.00,14.00,"377,122.46","377,122.46","33,684.94","33,684.94","11,313.41","11,313.41","33,684.94","33,684.94"
425113,814965B00_249118,35.00,30.00,"52,499.91","28,094.97",576.09,576.09,"3,292.88","1,708.03","16,108.95","9,321.12"


In [23]:
transaction_behavior_df["sent_received_ratio"] = transaction_behavior_df["total_amount_sent"]/(transaction_behavior_df["total_amount_received"] + 1)
transaction_behavior_df["transaction_direction_ratio"] = transaction_behavior_df["transaction_count_sent"]/(transaction_behavior_df["transaction_count_received"] + 1)

In [27]:
windows = {
    col.split("_")[-1]
    for col in time_behavior_df.columns
    if col.startswith(("total_amount_", "transaction_count_")) and col.count("_") >= 3
}

In [28]:
windows

{'30d', '7d', '90d'}

In [30]:
metrics = ["total_amount", "transaction_count"]
for metric in metrics:
    for time_window in windows:
        time_behavior_df[f"{metric}_{time_window}_ratio"] = time_behavior_df[f"{metric}_sent_{time_window}"] / (time_behavior_df[f"{metric}_received_{time_window}"] + 1e-6)

#### Join all features
---

In [31]:
transaction_behavior_df.head()

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,max_amount_sent,sent_received_ratio,transaction_direction_ratio
0,100428660_70,"1,084.00","168,672.00","479,594.79","52,762,291,209.75",238.44,"1,061.85",637.65,"8,093,751.98","5,551.91","1,615,849,280.83","110,014.08",155.46
1,1004286A8_70,653.00,"103,018.00","290,195.02","30,140,098,702.43",261.51,989.80,673.99,"6,795,155.33","4,971.15","954,757,442.23","103,861.17",157.52
2,1004286F0_70,108.00,"18,663.00","36,796.68","3,573,012,238.98",188.05,910.14,418.88,"2,869,372.33","2,117.38","165,255,167.69","97,098.85",171.22
3,100428738_70,98.00,"13,756.00","39,554.71","1,626,366,355.50",200.39,690.23,598.17,"2,625,842.87","2,410.15","187,245,987.35","41,115.84",138.95
4,100428780_70,117.00,"17,264.00","111,632.45","3,948,876,147.96",217.21,767.33,"4,371.38","4,548,156.67","33,667.82","286,863,871.15","35,373.59",146.31


In [32]:
time_behavior_df.head()

,customer_id,total_amount_received_7d,total_amount_received_30d,total_amount_received_90d,total_amount_sent_7d,total_amount_sent_30d,total_amount_sent_90d,transaction_count_received_7d,transaction_count_received_30d,transaction_count_received_90d,transaction_count_sent_7d,transaction_count_sent_30d,transaction_count_sent_90d,total_amount_7d_ratio,total_amount_90d_ratio,total_amount_30d_ratio,transaction_count_7d_ratio,transaction_count_90d_ratio,transaction_count_30d_ratio
0,100428660_70,"239,797.30","239,797.49",0,"23,722,077,156.41","29,040,214,053.34",0,539.00,545.00,0,"119,506.00","49,166.00",0,"98,925.54",0.00,"121,103.08",221.72,0.00,90.21
1,1004286A8_70,"145,097.47","145,097.55",0,"12,538,912,828.73","17,601,185,873.70",0,325.00,328.00,0,"72,881.00","30,137.00",0,"86,417.17",0.00,"121,305.88",224.25,0.00,91.88
2,1004286F0_70,"18,398.34","18,398.34",0,"1,869,690,970.86","1,703,321,268.11",0,53.00,55.00,0,"13,293.00","5,370.00",0,"101,622.82",0.00,"92,580.16",250.81,0.00,97.64
3,100428738_70,"19,777.35","19,777.37",0,"627,748,067.69","998,618,287.81",0,45.00,53.00,0,"9,767.00","3,989.00",0,"31,740.76",0.00,"50,492.99",217.04,0.00,75.26
4,100428780_70,"55,816.22","55,816.23",0,"1,558,511,055.71","2,390,365,092.25",0,56.00,61.00,0,"12,315.00","4,949.00",0,"27,922.19",0.00,"42,825.63",219.91,0.00,81.13


In [33]:
feature_df = transaction_behavior_df.merge(time_behavior_df, on="customer_id", how="left", validate="1:1")

In [34]:
feature_df.head()

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,max_amount_sent,sent_received_ratio,transaction_direction_ratio,total_amount_received_7d,total_amount_received_30d,total_amount_received_90d,total_amount_sent_7d,total_amount_sent_30d,total_amount_sent_90d,transaction_count_received_7d,transaction_count_received_30d,transaction_count_received_90d,transaction_count_sent_7d,transaction_count_sent_30d,transaction_count_sent_90d,total_amount_7d_ratio,total_amount_90d_ratio,total_amount_30d_ratio,transaction_count_7d_ratio,transaction_count_90d_ratio,transaction_count_30d_ratio
0,100428660_70,"1,084.00","168,672.00","479,594.79","52,762,291,209.75",238.44,"1,061.85",637.65,"8,093,751.98","5,551.91","1,615,849,280.83","110,014.08",155.46,"239,797.30","239,797.49",0,"23,722,077,156.41","29,040,214,053.34",0,539.00,545.00,0,"119,506.00","49,166.00",0,"98,925.54",0.00,"121,103.08",221.72,0.00,90.21
1,1004286A8_70,653.00,"103,018.00","290,195.02","30,140,098,702.43",261.51,989.80,673.99,"6,795,155.33","4,971.15","954,757,442.23","103,861.17",157.52,"145,097.47","145,097.55",0,"12,538,912,828.73","17,601,185,873.70",0,325.00,328.00,0,"72,881.00","30,137.00",0,"86,417.17",0.00,"121,305.88",224.25,0.00,91.88
2,1004286F0_70,108.00,"18,663.00","36,796.68","3,573,012,238.98",188.05,910.14,418.88,"2,869,372.33","2,117.38","165,255,167.69","97,098.85",171.22,"18,398.34","18,398.34",0,"1,869,690,970.86","1,703,321,268.11",0,53.00,55.00,0,"13,293.00","5,370.00",0,"101,622.82",0.00,"92,580.16",250.81,0.00,97.64
3,100428738_70,98.00,"13,756.00","39,554.71","1,626,366,355.50",200.39,690.23,598.17,"2,625,842.87","2,410.15","187,245,987.35","41,115.84",138.95,"19,777.35","19,777.37",0,"627,748,067.69","998,618,287.81",0,45.00,53.00,0,"9,767.00","3,989.00",0,"31,740.76",0.00,"50,492.99",217.04,0.00,75.26
4,100428780_70,117.00,"17,264.00","111,632.45","3,948,876,147.96",217.21,767.33,"4,371.38","4,548,156.67","33,667.82","286,863,871.15","35,373.59",146.31,"55,816.22","55,816.23",0,"1,558,511,055.71","2,390,365,092.25",0,56.00,61.00,0,"12,315.00","4,949.00",0,"27,922.19",0.00,"42,825.63",219.91,0.00,81.13


In [35]:
save_path = f"{ROOT_PATH}\\src\\data\\preprocessed_customer_level_features.csv"
feature_df.to_csv(save_path, index=False)

#### Checking correlation
---

In [ ]:
#plot a correalation heatmap using plotly
correlation_columns = list(feature_df.columns)
correlation_columns.remove("customer_id")
corr_matrix = feature_df[correlation_columns].corr()

fig = px.imshow(
    corr_matrix,
    color_continuous_scale="RdBu",   # diverging scale (best for correlations)
    zmin=-1,
    zmax=1,
    aspect="auto",
    text_auto=".2f",
    title="Correlation Heatmap – Transaction Behavior Features"
)

fig.update_layout(
    width=1400,
    height=1000,
    coloraxis_colorbar=dict(
        title="Correlation",
        ticks="outside"
    )
)

fig.show()

In [ ]:
correlation_columns

In [ ]:
feature_df.shape